In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2 — Clone your repo
!git clone https://github.com/bodasingiksheeraja317-svg/streamsense.git
%cd streamsense

Cloning into 'streamsense'...
remote: Enumerating objects: 3122, done.
remote: Counting objects: 100% (3122/3122), done.
remote: Compressing objects: 100% (3096/3096), done.
remote: Total 3122 (delta 19), reused 3115 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (3122/3122), 3.29 MiB | 12.09 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Filtering content: 100% (3000/3000), 108.40 MiB | 1.46 MiB/s, done.
/content/streamsense/streamsense


In [ ]:
# Diagnostic — run this before anything else
import os, zipfile

# Check 1: did the zip extract at all?
print("=== Contents of /content/streamsense/data/ ===")
data_path = '/content/streamsense/data'
if os.path.exists(data_path):
    for item in os.listdir(data_path):
        print(f"  {item}")
else:
    print("  /content/streamsense/data/ does not exist")

# Check 2: what is inside the zip?
zip_path = '/content/drive/MyDrive/data_raw.zip'
print(f"\n=== First 20 entries inside {zip_path} ===")
with zipfile.ZipFile(zip_path, 'r') as z:
    for name in list(z.namelist())[:20]:
        print(f"  {name}")

Streaming output truncated to the last 5000 lines.
  raw\down\90e72357_nohash_0.wav
  raw\left\f19c1390_nohash_4.wav
  raw\go\42f81601_nohash_1.wav
  raw\no\dfb6450b_nohash_0.wav
  raw\go\954f190f_nohash_0.wav
  raw\up\9e92ef0c_nohash_1.wav
  raw\go\e1936ce8_nohash_1.wav
  raw\no\283d7a53_nohash_0.wav
  raw\go\cd85758f_nohash_3.wav
  raw\off\b8874962_nohash_1.wav
  raw\up\2005ca25_nohash_0.wav
  raw\down\8523766b_nohash_0.wav
  raw\on\3a69f765_nohash_4.wav
  raw\down\d3831f6a_nohash_0.wav
  raw\no\cce7416f_nohash_10.wav
  raw\left\c93d5e22_nohash_3.wav
  raw\down\1ecfb537_nohash_1.wav
  raw\on\2b5e346d_nohash_0.wav
  raw\left\7d6b4b10_nohash_1.wav
  raw\yes\6969e51a_nohash_1.wav
  raw\no\f9643d42_nohash_1.wav
  raw\stop\e55a2b20_nohash_0.wav
  raw\go\2927c601_nohash_0.wav
  raw\off\b66f4f93_nohash_7.wav
  raw\no\686d030b_nohash_0.wav
  raw\down\9b5815cd_nohash_1.wav
  raw\go\3b3d2f59_nohash_0.wav
  raw\stop\327289eb_nohash_2.wav
  raw\up\7e7ca854_nohash_3.wav
  raw\left\74b73f88_nohash

In [ ]:
# Fix — extract directly to the correct location
import zipfile, os

zip_path    = '/content/drive/MyDrive/data_raw.zip'
extract_to  = '/content/streamsense/data/'

# Make sure target exists
os.makedirs(extract_to, exist_ok=True)

print("Extracting... this will take 2-3 minutes")
with zipfile.ZipFile(zip_path, 'r') as z:
    # Fix backslashes from Windows zip → forward slashes for Linux
    for member in z.infolist():
        # Convert Windows path separators
        member.filename = member.filename.replace('\\', '/')
        z.extract(member, extract_to)

print("\nDone. Checking:")
for cls in sorted(os.listdir('/content/streamsense/data/raw')):
    count = len(os.listdir(f'/content/streamsense/data/raw/{cls}'))
    print(f"  {cls}: {count} files")

Extracting... this will take 2-3 minutes

Done. Checking:
  down: 3917 files
  go: 3880 files
  left: 3801 files
  no: 3941 files
  off: 3745 files
  on: 3845 files
  right: 3778 files
  stop: 3872 files
  up: 3723 files
  yes: 4044 files


In [ ]:
# Cell 5 — Verify split files exist
from pathlib import Path

splits = [
    '/content/streamsense/data/splits/train_files.txt',
    '/content/streamsense/data/splits/val_files.txt',
    '/content/streamsense/data/splits/test_files.txt',
]
for s in splits:
    p = Path(s)
    lines = p.read_text().strip().splitlines()
    print(f"{p.name}: {len(lines)} lines — {'OK' if lines else 'EMPTY'}")

train_files.txt: 26984 lines — OK
val_files.txt: 5783 lines — OK
test_files.txt: 5779 lines — OK


In [ ]:
# Cell 5b — Rewrite split file paths for Colab Linux paths
import re

splits = [
    '/content/streamsense/data/splits/train_files.txt',
    '/content/streamsense/data/splits/val_files.txt',
    '/content/streamsense/data/splits/test_files.txt',
]

for split_path in splits:
    with open(split_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split('|')
        if len(parts) == 3:
            # Convert Windows path to Colab path
            win_path = parts[0].strip()
            # Extract class/filename from Windows path
            # e.g. C:\STREAMSENSE\data\raw\yes\file.wav
            #   -> /content/streamsense/data/raw/yes/file.wav
            colab_path = win_path.replace('C:\\STREAMSENSE\\', '/content/streamsense/')
            colab_path = colab_path.replace('\\', '/')
            new_lines.append(f"{colab_path} | {parts[1].strip()} | {parts[2].strip()}\n")
        else:
            new_lines.append(line)

    with open(split_path, 'w') as f:
        f.writelines(new_lines)

    print(f"Rewritten: {split_path.split('/')[-1]}")

print("All split files updated for Colab paths.")

Rewritten: train_files.txt
Rewritten: val_files.txt
Rewritten: test_files.txt
All split files updated for Colab paths.


In [ ]:
# Fix mel_pipeline.py paths for Colab
mel_path = '/content/streamsense/training/mel_pipeline.py'

with open(mel_path, 'r') as f:
    content = f.read()

# Replace Windows path with Colab path
content = content.replace(
    r'C:\STREAMSENSE\stats\normalization_stats.json',
    '/content/streamsense/stats/normalization_stats.json'
)

with open(mel_path, 'w') as f:
    f.write(content)

print("Fixed mel_pipeline.py paths.")

# Verify the fix
with open(mel_path, 'r') as f:
    for i, line in enumerate(f, 1):
        if 'normalization' in line.lower() or 'stats' in line.lower():
            print(f"  Line {i}: {line.rstrip()}")

Fixed mel_pipeline.py paths.
  Line 13: Loads normalization stats from:
  Line 14:     /content/streamsense/stats/normalization_stats.json
  Line 28: STATS_FILE = Path(r"/content/streamsense/stats/normalization_stats.json")
  Line 43: # ── Load normalization stats at import ────────────────────────────────────────
  Line 44: if not STATS_FILE.exists():
  Line 46:         f"Normalization stats not found: {STATS_FILE}\n"
  Line 47:         f"Run compute_normstats.py first."
  Line 50: with open(STATS_FILE, "r") as _f:
  Line 51:     _stats = json.load(_f)
  Line 53: GLOBAL_MEAN = float(_stats["global_mean"])
  Line 54: GLOBAL_STD  = float(_stats["global_std"])
  Line 57:     raise ValueError(f"global_std={GLOBAL_STD} is invalid — stats file may be corrupt.")
  Line 139:     # ── Step 7: global normalization ─────────────────────────────────────────


In [ ]:
# Cell 6 — Run training
!python training/train.py

STREAMSENSE — train.py

Device       : cuda
GPU          : Tesla T4
VRAM         : 15.6 GB
Epochs       : 30
Batch size   : 32
Learning rate: 0.001
Seed         : 42

Loading datasets...
[dataset] Loaded 'train_files.txt': 26984 samples  |  augment=True
[dataloader] 'train_files.txt': 26984 samples → 844 batches (batch=32, shuffle=True, workers=2)
[dataset] Loaded 'val_files.txt': 5783 samples  |  augment=False
[dataloader] 'val_files.txt': 5783 samples → 181 batches (batch=32, shuffle=False, workers=2)

Model        : StreamSenseNet
Parameters   : 295,786

────────────────────────────────────────────────────────────
Starting training — max 30 epochs
Early stopping patience: 8 epochs
Best checkpoint → /content/streamsense/checkpoints/best_model.pth
────────────────────────────────────────────────────────────

  Epoch   1  [ 200/844]  loss=2.3391
  Epoch   1  [ 400/844]  loss=2.1591
  Epoch   1  [ 600/844]  loss=2.0098
  Epoch   1  [ 800/844]  loss=1.7208
  Epoch   1  [ 844/844]  loss=1

In [ ]:
# Fix ReduceLROnPlateau verbose argument in train.py
train_path = '/content/streamsense/training/train.py'

with open(train_path, 'r') as f:
    content = f.read()

content = content.replace(
    """    scheduler = ReduceLROnPlateau(
        optimizer,
        mode     = "min",       # monitor val_loss
        factor   = 0.5,         # halve LR on plateau
        patience = 3,           # wait 3 epochs before reducing
        min_lr   = 1e-6,
        verbose  = True,
    )""",
    """    scheduler = ReduceLROnPlateau(
        optimizer,
        mode     = "min",       # monitor val_loss
        factor   = 0.5,         # halve LR on plateau
        patience = 3,           # wait 3 epochs before reducing
        min_lr   = 1e-6,
    )"""
)

with open(train_path, 'w') as f:
    f.write(content)

print("Fixed. Re-running training...")

Fixed. Re-running training...


In [ ]:
import shutil
from pathlib import Path

# Create Drive folder
Path('/content/drive/MyDrive/STREAMSENSE_outputs').mkdir(exist_ok=True)

# Save checkpoint and log
shutil.copy('/content/streamsense/checkpoints/best_model.pth',
            '/content/drive/MyDrive/STREAMSENSE_outputs/best_model.pth')
shutil.copy('/content/streamsense/checkpoints/training_log.csv',
            '/content/drive/MyDrive/STREAMSENSE_outputs/training_log.csv')

print("Saved to Drive:")
print("  best_model.pth")
print("  training_log.csv")

Saved to Drive:
  best_model.pth
  training_log.csv


In [ ]:
import subprocess

commands = [
    'git config user.email "bodasingiksheeraja317@gmail.com"',
    'git config user.name "bodasingiksheeraja317-svg"',
    'git add checkpoints/best_model.pth',
    'git add checkpoints/training_log.csv',
    'git commit -m "checkpoint: best_model from Colab T4"',
    'git push origin main'
]

for cmd in commands:
    result = subprocess.run(cmd, shell=True, cwd='/content/streamsense',
                           capture_output=True, text=True)
    print(f"$ {cmd}")
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)

$ git config user.email "bodasingiksheeraja317@gmail.com"
$ git config user.name "bodasingiksheeraja317-svg"
$ git add checkpoints/best_model.pth
The following paths are ignored by one of your .gitignore files:
checkpoints/best_model.pth
hint: Use -f if you really want to add them.
hint: Turn this message off by running
hint: "git config advice.addIgnoredFile false"

$ git add checkpoints/training_log.csv
$ git commit -m "checkpoint: best_model from Colab T4"
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   data/splits/test_files.txt
	modified:   data/splits/train_files.txt
	modified:   data/splits/val_files.txt
	modified:   training/mel_pipeline.py
	modified:   training/train.py

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
%%bash
cd /content/streamsense
git add -f checkpoints/best_model.pth
git add checkpoints/training_log.csv
echo "Added files"

Added files


In [ ]:
%%bash
cd /content/streamsense

# Stage everything needed
git add -f checkpoints/best_model.pth
git add checkpoints/training_log.csv
git add data/splits/train_files.txt
git add data/splits/val_files.txt
git add data/splits/test_files.txt
git add training/mel_pipeline.py
git add training/train.py

git commit -m "checkpoint: best_model Colab T4 + path fixes"

# Push with token embedded in URL
git push https://bodasingiksheeraja317-svg:<YOUR_NEW_TOKEN>@github.com/bodasingiksheeraja317-svg/streamsense.git main

On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


To https://github.com/bodasingiksheeraja317-svg/streamsense.git
   138efd7..ed9c7e3  main -> main


In [ ]:
%%bash
cd /content/streamsense

# Read current gitignore
cat .gitignore

data/raw/
*.wav
*.WAV
streamsense-env-win/
__pycache__/
*.pyc
checkpoints/*.pth
onnx_models/

In [ ]:
%%bash
cd /content/streamsense

# Remove the line blocking pth files
sed -i '/checkpoints\/\*.pth/d' .gitignore

git add .gitignore
git commit -m "fix: allow best_model.pth in git"
git push https://bodasingiksheeraja317-svg:<YOUR_NEW_TOKEN>@github.com/bodasingiksheeraja317-svg/streamsense.git main

On branch main
Your branch is ahead of 'origin/main' by 3 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


To https://github.com/bodasingiksheeraja317-svg/streamsense.git
   ed9c7e3..1213bda  main -> main
